
# Calculation of Outliers with Cluster Confidence Index

This is an outlier scoring method that compares the pathways of each streamline
in a bundle (pairwise) and scores each streamline by how many other streamlines
have similar pathways. The details can be found in :footcite:p:`Jordan2018`.


In [ ]:
import matplotlib.pyplot as plt

from dipy.core.gradients import gradient_table
from dipy.data import default_sphere, get_fnames
from dipy.direction import peaks_from_model
from dipy.io.gradients import read_bvals_bvecs
from dipy.io.image import load_nifti, load_nifti_data
from dipy.reconst.shm import CsaOdfModel
from dipy.tracking import utils
from dipy.tracking.stopping_criterion import ThresholdStoppingCriterion
from dipy.tracking.streamline import Streamlines, cluster_confidence
from dipy.tracking.tracker import eudx_tracking
from dipy.tracking.utils import length
from dipy.viz import actor, colormap, window

First, we need to generate some streamlines. For a more complete
description of these steps, please refer to the CSA Probabilistic Tracking
and the Visualization of ROI Surface Rendered with Streamlines Tutorials.



In [ ]:
hardi_fname, hardi_bval_fname, hardi_bvec_fname = get_fnames(name="stanford_hardi")
label_fname = get_fnames(name="stanford_labels")

data, affine = load_nifti(hardi_fname)
labels = load_nifti_data(label_fname)
bvals, bvecs = read_bvals_bvecs(hardi_bval_fname, hardi_bvec_fname)
gtab = gradient_table(bvals, bvecs=bvecs)

white_matter = (labels == 1) | (labels == 2)
csa_model = CsaOdfModel(gtab, sh_order_max=6)
csa_peaks = peaks_from_model(
    csa_model,
    data,
    default_sphere,
    relative_peak_threshold=0.8,
    min_separation_angle=45,
    mask=white_matter,
)
stopping_criterion = ThresholdStoppingCriterion(csa_peaks.gfa, 0.25)

We will use a slice of the anatomically-based corpus callosum ROI as our
seed mask to demonstrate the method.



In [ ]:
# Make a corpus callosum seed mask for tracking
seed_mask = labels == 2
seeds = utils.seeds_from_mask(seed_mask, affine, density=[1, 1, 1])
# Make a streamline bundle model of the corpus callosum ROI connectivity
streamlines_generator = eudx_tracking(
    seeds,
    stopping_criterion,
    affine,
    pam=csa_peaks,
    random_seed=1,
    sphere=default_sphere,
    step_size=2,
)
streamlines = Streamlines(streamlines_generator)

We do not want our results inflated by short streamlines, so we remove
streamlines shorter than 40mm prior to calculating the CCI.



In [ ]:
lengths = list(length(streamlines))
long_streamlines = Streamlines()
for i, sl in enumerate(streamlines):
    if lengths[i] > 40:
        long_streamlines.append(sl)

Now we calculate the Cluster Confidence Index using the corpus callosum
streamline bundle and visualize them.



In [ ]:
cci = cluster_confidence(long_streamlines)

We color each streamline by its CCI, going from pale cyan (low) to saturated
red (high): hue goes from 0.5 to 1 and saturation from 0 to 1. The color bar
maps values to these colors. They saturate above a quarter of the maximum CCI.



In [ ]:
cci_range = (cci.min(), cci.max() / 4)
cci_bar = actor.scalar_bar(
    clim=cci_range,
    hsv_range=((0.5, 1), (0, 1), (1, 1)),
    orientation="vertical",
    title="CCI",
    tick_format=".1f",
    position=(700, 250),
)
cci_colors = cci_bar.map_values(cci)

stream_actor = actor.streamlines(
    long_streamlines, colors=cci_colors, thickness=0.05, outline_thickness=0
)
scene = window.Scene()
scene.add(stream_actor)
scene.ui_scene.add(cci_bar)

Show the streamlines from above, FURY's default camera view, with an
orientation gizmo.



In [ ]:
show_manager = window.ShowManager(title="cci_streamlines", size=(800, 800), scene=scene)
actor.gizmo(show_manager)
show_manager.start()

.. rst-class:: centered small fst-italic fw-semibold

Cluster Confidence Index of corpus callosum dataset.


If you think of each streamline as a sample of a potential pathway through a
complex landscape of white matter anatomy probed via water diffusion,
intuitively we have more confidence that pathways represented by many samples
(streamlines) reflect a more stable representation of the underlying
phenomenon we are trying to model (anatomical landscape) than do lone
samples.

The CCI provides a voting system where by each streamline (within a set
tolerance) gets to vote on how much support it lends to. Outlier pathways
score relatively low on CCI, since they do not have many streamlines voting
for them. These outliers can be removed by thresholding on the CCI metric.



In [ ]:
fig, ax = plt.subplots(1)
ax.hist(cci, bins=100, histtype="step")
ax.set_xlabel("CCI")
ax.set_ylabel("# streamlines")
fig.savefig("cci_histogram.png")

.. rst-class:: centered small fst-italic fw-semibold

Histogram of Cluster Confidence Index values.


Now we threshold the CCI, defining outliers as streamlines that score
below 1.



In [ ]:
keep_streamlines = Streamlines()
for i, sl in enumerate(long_streamlines):
    if cci[i] >= 1:
        keep_streamlines.append(sl)

Show the filtered streamlines from above, FURY's default camera view, with an
orientation gizmo.



In [ ]:
keep_streamlines_actor = actor.streamlines(
    keep_streamlines,
    colors=colormap.line_colors(keep_streamlines),
    thickness=0.05,
    outline_thickness=0,
)
scene = window.Scene()
scene.add(keep_streamlines_actor)

show_manager = window.ShowManager(
    title="filtered_cci_streamlines", size=(800, 800), scene=scene
)
actor.gizmo(show_manager)
show_manager.start()

.. rst-class:: centered small fst-italic fw-semibold

Outliers, defined as streamlines scoring CCI < 1, were excluded.


## References

.. footbibliography::




.. include:: ../../links_names.inc


